# Preprocessing for Evaluation

In order to execute this notebook you need to upload a **finetuned model + tokenizer** (zip file). The notebook generates two **.tsv files** that can be used for further evaluation in **evaluation.ipynb**

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install transformers==4.51.3
    !pip install --no-deps unsloth

In [ ]:
from unsloth import FastLanguageModel
import re
from datasets import load_dataset, concatenate_datasets, Dataset
from transformers import pipeline
import nltk
from nltk.tokenize import word_tokenize
from tqdm import tqdm
import pandas as pd
import json
nltk.download('punkt_tab')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip drive/MyDrive/lora_en_tox_e3_75.zip -d lora

Archive:  drive/MyDrive/lora_en_tox_e3_75.zip
   creating: lora/lora/
  inflating: lora/lora/tokenizer.json  
  inflating: lora/lora/adapter_config.json  
  inflating: lora/lora/README.md     
  inflating: lora/lora/special_tokens_map.json  
  inflating: lora/lora/tokenizer_config.json  
  inflating: lora/lora/added_tokens.json  
  inflating: lora/lora/adapter_model.safetensors  
  inflating: lora/lora/vocab.json    
  inflating: lora/lora/merges.txt    


In [ ]:
system_prompt = """
You are an advanced language model tasked with detoxifying texts across multiple languages.
Your goal is to identify and neutralize toxic content while preserving the original meaning and language of the input text.

## Definition of Toxicity

Toxic language includes expressions that are offensive, hateful, abusive, discriminatory, or threatening.
It can be explicit, such as direct insults or slurs, or implicit, such as passive-aggressive comments or coded language.
Toxicity reduces the quality of conversation and harms communities.

## Steps to Success

1. Identify toxic content in the text while considering its context. There is also a list of the most probable toxic words given.
2. Reformulate the text so it becomes a non-toxic text in the initial language
3. Check if it still contains toxic content
4. Make sure that the semantic content of the non-toxic text remains as similar as possible to the original text
5. Output only the detoxified final text

## Examples of Toxic Input and Non-Toxic Output Text Pairs in English

Toxic: "then all of a sudden i see her , shes now got the big phony tits and everything." ["tits"]
Non-toxic: "All of a sudden i see her, she is all grown up."

Toxic: "i would have lost my shit * years * ago." ["shit"]
Non-toxic: "I would have gone crazy years ago."

Toxic: "How about you fuck off and don't stalk my edits?" ["fuck"]
Non-toxic: "How about you leave me alone and don't stalk my edits?"

Toxic: "You dumb motherfucker I am going to sue the shit out of you" ["dumb", "motherfucker", "shit"]
Non-toxic: "I am going to sue everything out of you"

## Task

Please detoxify the following toxic text using all the things provided above:
"""

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora",
    max_seq_length = 2048,
    load_in_4bit = True,
)
model.eval()

In [ ]:
def extract_cleaned(decoded_output):
    matches = re.findall(r'<think>\s*</think>\s*\n*\s*(.*?)$', decoded_output.strip(), re.DOTALL)
    cleaned = matches[-1].strip() if matches else ''
    cleaned = re.sub(r'^[\+\-\•\#\*\~\s]+', '', cleaned)
    return cleaned

def template(problem, toxic_tokens):
    return [
        {"role": "user", "content": system_prompt + problem + " " + toxic_tokens}
    ]

In [ ]:
dataset = load_dataset("textdetox/multilingual_paradetox")
#dataset['en'] = dataset['en'].map(lambda x:{"lang": 'en'})
dataset['de'] = dataset['de'].map(lambda x:{"lang": 'de'})
dataset['ru'] = dataset['ru'].map(lambda x:{"lang": 'ru'})
dataset['es'] = dataset['es'].map(lambda x:{"lang": 'es'})
dataset['uk'] = dataset['uk'].map(lambda x:{"lang": 'uk'})
dataset['zh'] = dataset['zh'].map(lambda x:{"lang": 'zh'})

ds = concatenate_datasets([dataset["de"], dataset["ru"], dataset["es"], dataset["uk"], dataset["zh"]], axis=0)
#ds = dataset['en']

In [ ]:
eval_data = ds

In [ ]:
en_classifier = pipeline("text-classification", model="unitary/toxic-bert", device=0)  # falls GPU

def get_toxic_words_batched(sentence, threshold=0.45):
    sentence = re.sub(r'[^\w\s]', '', sentence)
    toks = word_tokenize(sentence)
    if not toks:
        return []

    results = en_classifier(toks, batch_size=16)
    return [tok for tok, res in zip(toks, results) if res["score"] >= threshold]

In [ ]:
tqdm.pandas()
df = pd.DataFrame(eval_data)
df["toxic_tokens"] = df["toxic_sentence"].progress_apply(get_toxic_words_batched)
eval_data = Dataset.from_pandas(df)

100%|██████████| 2000/2000 [00:26<00:00, 74.21it/s] 


In [ ]:
eval_data = eval_data.map(lambda example: {
    "toxic_tokens": json.dumps(example["toxic_tokens"])
})

In [ ]:

split = eval_data.shuffle(seed = 3407).train_test_split(test_size=0.25, seed=42)
combined_dataset = split["train"]
test_dataset = split["test"]

combined_dataset, test_dataset

(Dataset({
     features: ['toxic_sentence', 'neutral_sentence', 'lang', 'toxic_tokens'],
     num_rows: 1500
 }),
 Dataset({
     features: ['toxic_sentence', 'neutral_sentence', 'lang', 'toxic_tokens'],
     num_rows: 500
 }))

In [ ]:
eval_data = test_dataset

In [ ]:
def create_tsv(model, tokenizer, eval_data):
  print(len(eval_data))
  reference_df = eval_data.to_pandas()
  reference_df.to_csv("reference.tsv", sep="\t", index=False)

  evaluation_df = reference_df.copy()
  evaluation_df['neutral_sentence'] = ""
  print(len(evaluation_df))

  input = [
    template(row["toxic_sentence"], row["toxic_tokens"])
    for _, row in evaluation_df.iterrows()
  ]
  detoxified = []
  len_input = len(input)
  for i in range(0, len_input, 1):
    print(f"{i}/{len_input}")
    text = tokenizer.apply_chat_template(
        input[i],
        tokenize = False,
        add_generation_prompt = True,
        enable_thinking = False,
    )
    outputs = model.generate(
        **tokenizer(text, return_tensors = "pt").to("cuda"),
        max_new_tokens = 256,
        temperature = 0.7, top_p = 0.8, top_k = 20,
    )
    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    output_cleaned = extract_cleaned(output_text)
    print(output_cleaned)
    detoxified.append(output_cleaned)
  evaluation_df["neutral_sentence"] = detoxified
  evaluation_df.to_csv('lora.tsv', sep="\t", index=False, encoding="utf-8", errors='backslashreplace')

  return evaluation_df, reference_df

In [ ]:
evaluation_df, reference_df = create_tsv(model, tokenizer, eval_data) #<-----------

500
500
0/500
For me, there is a kind of universal dislike.
1/500
In einer nicht #PoliticalCorrectness-geistesgestörten Welt würden sie dem #Invasoren-#Abschaum die Scheiße aus dem Hirn prügeln
2/500
Tu eres un poco raro, punto.
3/500
El gobierno, debería dejar de hablar sin sentido contra toda evidencia.
4/500
и на этом закончить с тобой беседу видно чмо капитальное
5/500
Little fairies should be raised from a young age.
6/500
Todenhöfer is a liar leftist Islam lobbyist.  Unfortunately very influential
7/500
Сами превращаете интернет в помойку, сами же за это и получаете.
8/500
Das Fette Schwein @USER_0 Gabriel Redet wieder auf ,alles vor der Wahl 2017 😂😂😂😂😂
9/500
це коли карент муд від []
10/500
En una cárcel española cuesta dinero, en su casa en Marruecos no cuesta nada.
11/500
Trump ist nicht eingeknickt und schon gar nicht vor diesen Politikern, Lügenpresse und Nazimedien in Deutschland!!
12/500
Вы такие ненормальные, такое писать? Мне интересно, в каком месте нужно смеяться? Не т